In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import minimize
import os
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
import xgboost as xgb

In [ ]:
DATA_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'v12' else os.getcwd()
xlsx = pd.ExcelFile(os.path.join(DATA_DIR, 'Data for Datathon (Revised).xlsx'))
template = pd.read_csv(os.path.join(DATA_DIR, 'template_forecast_v00.csv'))

portfolios = ['A','B','C','D']

daily = {}
for p in portfolios:
    df = pd.read_excel(xlsx, f'{p} - Daily')
    df['Date'] = pd.to_datetime(df['Date'].str.strip().str.rsplit(' ', n=1).str[0], format='%m/%d/%y')
    df = df.sort_values('Date').reset_index(drop=True)
    df.columns = [c.strip() for c in df.columns]
    daily[p] = df
print({p: len(daily[p]) for p in portfolios})

mmap = {'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
        'July':7,'August':8,'September':9,'October':10,'November':11,'December':12}
intervals = {}
for p in portfolios:
    df = pd.read_excel(xlsx, f'{p} - Interval')
    df.columns = [c.strip() for c in df.columns]
    df = df.dropna(subset=['Interval']).copy()
    df['mnum'] = df['Month'].map(mmap)
    df['Day'] = df['Day'].astype(int)
    df['Date'] = pd.to_datetime(dict(year=2024, month=df['mnum'], day=df['Day']))
    df['slot'] = df['Interval'].apply(lambda t: t.hour*2 + t.minute//30)
    df = df.sort_values(['Date','slot']).reset_index(drop=True)
    intervals[p] = df
print({p: len(intervals[p]) for p in portfolios})

staff = pd.read_excel(xlsx, 'Daily Staffing')
staff.columns = ['Date'] + [f'Staff_{p}' for p in portfolios]
staff['Date'] = pd.to_datetime(staff['Date'])
staff = staff.sort_values('Date').reset_index(drop=True)
print(f'staffing: {len(staff)} days')

In [ ]:
# enhanced features — adds days_since/until_holiday, friday flag, dom cyclical,
# rolling std, calls_per_agent, agents_change from train.py
holidays = pd.to_datetime(['2024-01-01','2024-01-15','2024-02-19','2024-05-27','2024-06-19',
    '2024-07-04','2024-09-02','2024-10-14','2024-11-11','2024-11-28','2024-12-25',
    '2025-01-01','2025-01-20','2025-02-17','2025-05-26','2025-06-19',
    '2025-07-04','2025-09-01','2025-10-13','2025-11-11','2025-11-27','2025-12-25'])

def make_features(df, port):
    f = pd.DataFrame()
    f['Date'] = df['Date']
    f['dow'] = df['Date'].dt.dayofweek
    f['dom'] = df['Date'].dt.day
    f['month'] = df['Date'].dt.month
    f['woy'] = df['Date'].dt.isocalendar().week.astype(int)
    f['year'] = df['Date'].dt.year
    f['quarter'] = df['Date'].dt.quarter
    f['wknd'] = (f['dow'] >= 5).astype(int)
    f['mon'] = (f['dow'] == 0).astype(int)
    f['fri'] = (f['dow'] == 4).astype(int)
    
    # cyclical encoding
    f['dow_s'] = np.sin(2*np.pi*f['dow']/7)
    f['dow_c'] = np.cos(2*np.pi*f['dow']/7)
    f['month_s'] = np.sin(2*np.pi*f['month']/12)
    f['month_c'] = np.cos(2*np.pi*f['month']/12)
    f['dom_s'] = np.sin(2*np.pi*f['dom']/31)
    f['dom_c'] = np.cos(2*np.pi*f['dom']/31)
    
    f['holiday'] = df['Date'].isin(holidays).astype(int)
    f['month_start'] = (f['dom'] <= 5).astype(int)
    f['month_end'] = (f['dom'] >= 26).astype(int)
    
    # days since/until holiday
    holiday_arr = holidays.values
    ds_list, du_list = [], []
    for d in df['Date'].values:
        past = holiday_arr[holiday_arr <= d]
        future = holiday_arr[holiday_arr >= d]
        ds = (d - past[-1]) / np.timedelta64(1, 'D') if len(past) > 0 else 365
        du = (future[0] - d) / np.timedelta64(1, 'D') if len(future) > 0 else 365
        ds_list.append(ds)
        du_list.append(du)
    f['days_since_hol'] = ds_list
    f['days_until_hol'] = du_list
    
    # lags + rolling stats
    for m in ['Call Volume','CCT','Abandon Rate']:
        if m not in df.columns: continue
        f[f'{m}_l7'] = df[m].shift(7)
        f[f'{m}_l14'] = df[m].shift(14)
        f[f'{m}_l28'] = df[m].shift(28)
        f[f'{m}_l365'] = df[m].shift(365)
        f[f'{m}_r7'] = df[m].rolling(7).mean()
        f[f'{m}_r14'] = df[m].rolling(14).mean()
        f[f'{m}_r30'] = df[m].rolling(30).mean()
        f[f'{m}_std7'] = df[m].rolling(7).std()
        f[f'{m}_ew'] = df[m].ewm(span=7).mean()
    
    # staffing
    sc = f'Staff_{port}'
    f = f.merge(staff[['Date',sc]].rename(columns={sc:'agents'}), on='Date', how='left')
    f['agents_change'] = f['agents'].diff()
    
    # calls per agent (staffing pressure)
    if 'Call Volume' in df.columns:
        f['cv_per_agent'] = df['Call Volume'].values / f['agents'].replace(0, np.nan)
    
    for m in ['Call Volume','CCT','Abandon Rate']:
        if m in df.columns:
            f[f'tgt_{m}'] = df[m]
    return f

feats = {}
for p in portfolios:
    feats[p] = make_features(daily[p], p)
print({p: feats[p].shape for p in portfolios})

In [ ]:
# direct interval models for CV and CCT + profiles for ABD
interval_models = {}
cct_interval_models = {}
abd_prof = {}
cct_prof = {}

for p in portfolios:
    df = intervals[p].copy()
    df['dow'] = df['Date'].dt.dayofweek
    dtot = df.groupby('Date')['Call Volume'].sum().reset_index()
    dtot.columns = ['Date','daily_cv']
    df = df.merge(dtot, on='Date')
    
    # daily CCT average for interval model
    dcct_tot = df.groupby('Date')['CCT'].mean().reset_index()
    dcct_tot.columns = ['Date','daily_cct']
    df = df.merge(dcct_tot, on='Date')
    
    df['hour'] = df['slot'] // 2
    df['is_peak'] = ((df['hour']>=9)&(df['hour']<=17)).astype(int)
    df['slot_sin'] = np.sin(2*np.pi*df['slot']/48)
    df['slot_cos'] = np.cos(2*np.pi*df['slot']/48)
    df['dow_sin'] = np.sin(2*np.pi*df['dow']/7)
    df['dow_cos'] = np.cos(2*np.pi*df['dow']/7)
    df['dom'] = df['Date'].dt.day
    
    icols = ['slot','dow','daily_cv','hour','is_peak','slot_sin','slot_cos','dow_sin','dow_cos','dom']
    clean = df.dropna(subset=['Call Volume'])
    
    # CV interval model
    gb = HistGradientBoostingRegressor(max_iter=300, max_depth=5,
        learning_rate=0.05, min_samples_leaf=5, random_state=42)
    gb.fit(clean[icols].values, clean['Call Volume'].values)
    interval_models[p] = {'model': gb, 'cols': icols}
    print(f'{p}: CV interval model trained on {len(clean)} rows')
    
    # CCT interval model
    icols_cct = ['slot','dow','daily_cct','hour','is_peak','slot_sin','slot_cos','dow_sin','dow_cos','dom']
    clean_cct = df.dropna(subset=['CCT'])
    gb_cct = HistGradientBoostingRegressor(max_iter=300, max_depth=5,
        learning_rate=0.05, min_samples_leaf=5, random_state=42)
    gb_cct.fit(clean_cct[icols_cct].values, clean_cct['CCT'].values)
    cct_interval_models[p] = {'model': gb_cct, 'cols': icols_cct}
    print(f'{p}: CCT interval model trained on {len(clean_cct)} rows')
    
    # ABD profiles (keep profile-based since ABD data is sparse)
    abt = df.groupby('Date')['Abandoned Calls'].transform('sum')
    df['abd_pct'] = df['Abandoned Calls'] / abt.replace(0, np.nan)
    abd_prof[p], cct_prof[p] = {}, {}
    for dow in range(7):
        sub = df[df['dow']==dow]
        pr = sub.groupby('slot')['abd_pct'].median()
        a = np.zeros(48); a[pr.index.astype(int)] = pr.values
        a = np.nan_to_num(a, 0); a = gaussian_filter1d(a, sigma=0.7)
        if a.sum() > 0: a /= a.sum()
        abd_prof[p][dow] = a
        # keep CCT profiles as fallback
        pr = sub.groupby('slot')['CCT'].median()
        a = np.zeros(48); a[pr.index.astype(int)] = pr.values
        a = np.nan_to_num(a, 0); a = gaussian_filter1d(a, sigma=0.7)
        cct_prof[p][dow] = a

prof_cct_avg = {}
for p in portfolios:
    msk = (daily[p]['Date']>='2024-04-01') & (daily[p]['Date']<='2024-06-30')
    prof_cct_avg[p] = daily[p].loc[msk, 'CCT'].mean()
print('done')

In [ ]:
# asymmetric loss for validation — understaffing penalized more
def asym_loss(y_true, y_pred, alpha=2.0):
    err = y_true - y_pred  # positive = underprediction
    return np.mean(np.where(err > 0, alpha * err**2, err**2))

def feat_cols(df):
    skip = ['Date','tgt_Call Volume','tgt_CCT','tgt_Abandon Rate']
    return [c for c in df.columns if c not in skip]

preds = {}
val_preds = {}  # store validation predictions for bias calibration
val_actuals = {}
aug = pd.date_range('2025-08-01','2025-08-31')

for p in portfolios:
    print(f'\n--- {p} ---')
    ft = feats[p]
    cols = feat_cols(ft)
    ok = ft[cols].notna().all(axis=1)
    cl = ft[ok].copy()
    
    trn = cl['Date'] < '2025-07-01'
    val = (cl['Date']>='2025-07-01') & (cl['Date']<'2025-08-01')
    Xtr, Xv = cl.loc[trn, cols].values, cl.loc[val, cols].values
    
    preds[p] = {}
    val_preds[p] = {}
    val_actuals[p] = {}
    d = daily[p]
    a24 = d[(d['Date'].dt.month==8)&(d['Date'].dt.year==2024)]
    
    for met in ['Call Volume','CCT']:
        ytr = cl.loc[trn, f'tgt_{met}'].values
        yv = cl.loc[val, f'tgt_{met}'].values
        val_actuals[p][met] = yv
        
        # higher quantiles — understaffing penalized more
        q = 0.58 if met=='CCT' else 0.65
        gb = HistGradientBoostingRegressor(loss='quantile', quantile=q,
            max_iter=500, max_depth=6, learning_rate=0.05,
            min_samples_leaf=10, random_state=42)
        gb.fit(Xtr, ytr)
        
        rd = Ridge(alpha=1.0)
        rd.fit(np.nan_to_num(Xtr,0), ytr)
        
        # XGBoost with quantile loss
        try:
            xgb_model = xgb.XGBRegressor(
                objective='reg:quantileerror',
                quantile_alpha=q,
                n_estimators=500,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                verbosity=0
            )
            xgb_model.fit(np.nan_to_num(Xtr,0), ytr)
            has_xgb = True
        except Exception:
            # fallback if quantile not supported
            xgb_model = xgb.XGBRegressor(
                objective='reg:squarederror',
                n_estimators=500,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                verbosity=0
            )
            xgb_model.fit(np.nan_to_num(Xtr,0), ytr)
            has_xgb = True
        
        # 3-model ensemble: 40% HGB + 35% XGB + 25% Ridge
        vp = (0.40*gb.predict(Xv) + 
              0.35*xgb_model.predict(np.nan_to_num(Xv,0)) + 
              0.25*rd.predict(np.nan_to_num(Xv,0)))
        
        print(f'  {met} MAE: {mean_absolute_error(yv,vp):.2f}  AsymLoss: {asym_loss(yv,vp):.1f}')
        val_preds[p][met] = vp
        
        # august pred
        amsk = (ft['Date']>='2025-08-01') & (ft['Date']<='2025-08-31')
        Xa = ft.loc[amsk, cols].fillna(method='ffill').fillna(method='bfill').fillna(0)
        ml = (0.40*gb.predict(Xa.values) + 
              0.35*xgb_model.predict(np.nan_to_num(Xa.values,0)) + 
              0.25*rd.predict(np.nan_to_num(Xa.values,0)))
        
        # baseline from last aug
        g24 = d[(d['Date'].dt.year==2024)&(d['Date'].dt.month<=7)][met].mean()
        g25 = d[(d['Date'].dt.year==2025)&(d['Date'].dt.month<=7)][met].mean()
        gr = g25/g24 if g24>0 else 1.0
        bl = np.zeros(31)
        for i,dt in enumerate(aug):
            m = a24[a24['Date'].dt.dayofweek==dt.dayofweek][met].values
            bl[i] = m.mean()*gr if len(m)>0 else a24[met].mean()*gr
        
        final = 0.7*ml[:31] + 0.3*bl
        preds[p][met] = final
        print(f'  aug avg: {final.mean():.1f}')
    
    # abandon rate — log-transform ML + heuristic hybrid
    recent = d[(d['Date']>='2025-06-01')&(d['Date']<'2025-08-01')]
    
    # heuristic baseline (same as before)
    abd_heuristic = np.zeros(31)
    for i,dt in enumerate(aug):
        dw = dt.dayofweek
        r = recent[recent['Date'].dt.dayofweek==dw]['Abandon Rate']
        a = a24[a24['Date'].dt.dayofweek==dw]['Abandon Rate']
        if len(r)>0 and len(a)>0:
            abd_heuristic[i] = 0.6*r.mean() + 0.4*a.mean()
        elif len(r)>0:
            abd_heuristic[i] = r.mean()
        else:
            abd_heuristic[i] = d['Abandon Rate'].tail(60).mean()
    abd_heuristic *= 1.1
    
    # ML attempt with log-transform
    try:
        eps = 0.001
        ytr_abd = np.log(cl.loc[trn, 'tgt_Abandon Rate'].values + eps)
        yv_abd = cl.loc[val, 'tgt_Abandon Rate'].values
        gb_abd = HistGradientBoostingRegressor(max_iter=300, max_depth=4,
            learning_rate=0.03, min_samples_leaf=15, random_state=42)
        gb_abd.fit(Xtr, ytr_abd)
        abd_ml_val = np.exp(gb_abd.predict(Xv)) - eps
        abd_ml_val = np.clip(abd_ml_val, 0, 0.25)
        ml_mae = mean_absolute_error(yv_abd, abd_ml_val)
        heur_mae = mean_absolute_error(yv_abd, abd_heuristic[:len(yv_abd)] if len(yv_abd)<=31 else abd_heuristic)
        print(f'  ABD ML MAE: {ml_mae:.4f} vs Heuristic MAE: {heur_mae:.4f}')
        
        # august ML prediction
        amsk = (ft['Date']>='2025-08-01') & (ft['Date']<='2025-08-31')
        Xa_abd = ft.loc[amsk, cols].fillna(method='ffill').fillna(method='bfill').fillna(0)
        abd_ml = np.exp(gb_abd.predict(Xa_abd.values)) - eps
        abd_ml = np.clip(abd_ml[:31], 0, 0.25)
        
        # hybrid: 40% ML + 60% heuristic (heuristic still gets more weight for stability)
        abd = 0.4*abd_ml + 0.6*abd_heuristic
        print(f'  ABD: using hybrid (40% ML + 60% heuristic)')
    except Exception as e:
        print(f'  ABD ML failed ({e}), using heuristic only')
        abd = abd_heuristic
    
    abd = np.clip(abd, 0.002, 0.25)
    preds[p]['Abandon Rate'] = abd
    abd_avg = a24['Abandon Rate'].mean()
    print(f'  ABD: {abd.mean():.4f} (aug24 was {abd_avg:.4f})')

print('\ndone')

In [ ]:
# disaggregate using direct interval models for CV and CCT, profiles for ABD
aug_dates = pd.date_range('2025-08-01','2025-08-31')
res = {p: {'cv':[],'abd':[],'ar':[],'cct':[]} for p in portfolios}

for p in portfolios:
    dcv = preds[p]['Call Volume']
    dcct = preds[p]['CCT']
    dar = preds[p]['Abandon Rate']
    dabd = dcv * dar
    cv_model = interval_models[p]['model']
    cct_model = cct_interval_models[p]['model']
    
    for i,dt in enumerate(aug_dates):
        dw = dt.dayofweek
        dom = dt.day
        rows_cv = []
        rows_cct = []
        for slot in range(48):
            hour = slot // 2
            is_peak = 1 if 9 <= hour <= 17 else 0
            ss, sc = np.sin(2*np.pi*slot/48), np.cos(2*np.pi*slot/48)
            ds, dc = np.sin(2*np.pi*dw/7), np.cos(2*np.pi*dw/7)
            rows_cv.append([slot, dw, dcv[i], hour, is_peak, ss, sc, ds, dc, dom])
            rows_cct.append([slot, dw, dcct[i], hour, is_peak, ss, sc, ds, dc, dom])
        
        # CV: direct interval model, rescaled to daily total
        X_cv = np.array(rows_cv)
        cv = np.clip(cv_model.predict(X_cv), 0, None)
        if cv.sum() > 0:
            cv = cv * (dcv[i] / cv.sum())
        
        # CCT: direct interval model
        X_cct = np.array(rows_cct)
        cc = np.clip(cct_model.predict(X_cct), 0, None)
        # rescale so weighted avg (by cv) matches daily CCT
        if cv.sum() > 0 and cc.sum() > 0:
            weighted_avg = np.average(cc, weights=np.maximum(cv, 1e-6))
            if weighted_avg > 0:
                cc = cc * (dcct[i] / weighted_avg)
        
        ab = dabd[i] * abd_prof[p][dw]
        ar = np.where(cv>0, ab/cv, 0.0)
        res[p]['cv'].append(cv)
        res[p]['abd'].append(ab)
        res[p]['ar'].append(ar)
        res[p]['cct'].append(cc)

for p in portfolios:
    for k in res[p]:
        res[p][k] = np.array(res[p][k])

In [ ]:
# bias from v10 (best performing)
bias = {'A': 1.13, 'B': 1.08, 'C': 1.06, 'D': 1.11}
print('Using v10 bias:', bias)

for p in portfolios:
    res[p]['cv'] *= bias[p]
    res[p]['abd'] *= bias[p]

for p in portfolios:
    res[p]['cv'] = np.clip(res[p]['cv'], 0, None)
    res[p]['abd'] = np.clip(res[p]['abd'], 0, None)
    res[p]['cct'] = np.clip(res[p]['cct'], 0, None)
    bad = res[p]['abd'] > res[p]['cv']
    res[p]['abd'][bad] = res[p]['cv'][bad]
    cv, ab = res[p]['cv'], res[p]['abd']
    res[p]['ar'] = np.clip(np.where(cv>0, ab/cv, 0.0), 0, 1)
    res[p]['cv'] = np.round(cv).astype(int)
    res[p]['abd'] = np.round(res[p]['abd']).astype(int)

In [ ]:
# save
rows = []
for day in range(31):
    for slot in range(48):
        h, m = slot//2, (slot%2)*30
        row = {'Month':'August', 'Day':str(day+1), 'Interval':f'{h}:{m:02d}'}
        for p in portfolios:
            row[f'Calls_Offered_{p}'] = int(res[p]['cv'][day,slot])
            row[f'Abandoned_Calls_{p}'] = int(res[p]['abd'][day,slot])
            row[f'Abandoned_Rate_{p}'] = round(float(res[p]['ar'][day,slot]), 6)
            row[f'CCT_{p}'] = round(float(res[p]['cct'][day,slot]), 2)
        rows.append(row)

sub = pd.DataFrame(rows)[template.columns.tolist()]
out = os.path.join(os.getcwd(), 'forecast_v12.csv')
sub.to_csv(out, index=False)

assert sub.shape == (1488, 19)
assert not sub.isnull().any().any()
for p in portfolios:
    assert (sub[f'Calls_Offered_{p}']>=0).all()
    assert (sub[f'Abandoned_Calls_{p}']<=sub[f'Calls_Offered_{p}']).all()
print('all good')

for p in portfolios:
    a24 = daily[p][(daily[p]['Date'].dt.month==8)&(daily[p]['Date'].dt.year==2024)]
    fc = sub[f'Calls_Offered_{p}'].sum()
    ac = a24['Call Volume'].sum()
    print(f'{p}: {fc:,} vs {ac:,.0f} ({fc/ac:.3f})')